In [1]:
import random
import datetime
from faker import Faker

fake = Faker('zh_CN')
random.seed(42)

# ==================== 日期配置（覆盖7月、8月、9月1-4日） ====================
NUM_SKU = 50
NUM_WAREHOUSE = 3
NUM_DAYS = 66  # 从 2026-07-01 到 2026-09-04 共 66 天

SNAPSHOT_DATE = datetime.date(2026, 9, 4)        # 当前快照日期（今天）
END_DATE_HIST = datetime.date(2026, 9, 4)        # 历史数据最后一天（包含今天）
START_DATE = datetime.date(2026, 7, 1)           # 历史数据起始日（7月1日）

sql_lines = []
def add_sql(line):
    sql_lines.append(line + ';')

def d2s(date_obj):
    return date_obj.strftime('%Y-%m-%d')

def dt2s(dt_obj):
    return dt_obj.strftime('%Y-%m-%d %H:%M:%S')

# ==================== 1. 品类维度 ====================
categories = [
    (101, '饮料', 100, 0, 1, '饮料', '0', 1, '2024-01-01', '9999-12-31', 1),
    (102, '食品', 200, 0, 1, '食品', '0', 1, '2024-01-01', '9999-12-31', 1),
    (103, '日化', 300, 0, 1, '日化', '0', 1, '2024-01-01', '9999-12-31', 1),
    (10101, '饮用水', 110, 101, 2, '饮料 / 饮用水', '0,101', 1, '2024-01-01', '9999-12-31', 1),
    (10102, '碳酸饮料', 120, 101, 2, '饮料 / 碳酸饮料', '0,101', 1, '2024-01-01', '9999-12-31', 1),
    (10201, '饼干', 210, 102, 2, '食品 / 饼干', '0,102', 1, '2024-01-01', '9999-12-31', 1),
    (10202, '方便面', 220, 102, 2, '食品 / 方便面', '0,102', 1, '2024-01-01', '9999-12-31', 1),
    (10301, '洗发水', 310, 103, 2, '日化 / 洗发水', '0,103', 1, '2024-01-01', '9999-12-31', 1),
    (10302, '洗衣液', 320, 103, 2, '日化 / 洗衣液', '0,103', 1, '2024-01-01', '9999-12-31', 1),
    (1010101, '瓶装矿泉水', 111, 10101, 3, '饮料 / 饮用水 / 瓶装矿泉水', '0,101,10101', 1, '2024-01-01', '9999-12-31', 1),
    (1010102, '桶装水', 112, 10101, 3, '饮料 / 饮用水 / 桶装水', '0,101,10101', 1, '2024-01-01', '9999-12-31', 1),
    (1010201, '可乐', 121, 10102, 3, '饮料 / 碳酸饮料 / 可乐', '0,101,10102', 1, '2024-01-01', '9999-12-31', 1),
    (1020101, '苏打饼干', 211, 10201, 3, '食品 / 饼干 / 苏打饼干', '0,102,10201', 1, '2024-01-01', '9999-12-31', 1),
    (1020201, '红烧牛肉面', 221, 10202, 3, '食品 / 方便面 / 红烧牛肉面', '0,102,10202', 1, '2024-01-01', '9999-12-31', 1),
    (1030101, '去屑洗发水', 311, 10301, 3, '日化 / 洗发水 / 去屑洗发水', '0,103,10301', 1, '2024-01-01', '9999-12-31', 1),
    (1030201, '洗衣液3kg', 321, 10302, 3, '日化 / 洗衣液 / 洗衣液3kg', '0,103,10302', 1, '2024-01-01', '9999-12-31', 1),
]
for c in categories:
    add_sql(f"INSERT INTO dim_category (category_id, category_name, category_code, parent_id, category_level, full_path_name, full_path_id, sort_no, start_date, end_date, is_valid) VALUES {c}")

# ==================== 2. 供应商维度 ====================
suppliers = [
    (1, '农夫山泉华东供应商', '浙江杭州', 50000, 'A', 1),
    (2, '统一食品华南供应商', '广东广州', 30000, 'B', 1),
    (3, '宝洁日化华北供应商', '天津', 40000, 'A', 1),
    (4, '蒙牛乳业西南供应商', '四川成都', 25000, 'B', 1),
    (5, '康师傅华东供应商', '江苏苏州', 35000, 'C', 1),
]
for s in suppliers:
    add_sql(f"INSERT INTO dim_supplier (supplier_id, supplier_name, region, max_capacity, quality_level, cooperation_status) VALUES {s}")

# ==================== 3. 仓库维度 ====================
warehouses = [
    (1, '华东RDC', 'RDC', '华东', '上海市嘉定区', 1),
    (2, '华南RDC', 'RDC', '华南', '广州市黄埔区', 1),
    (3, '华北RDC', 'RDC', '华北', '天津市武清区', 1),
]
for w in warehouses:
    add_sql(f"INSERT INTO dim_warehouse (warehouse_id, warehouse_name, warehouse_type, region, address, warehouse_status) VALUES {w}")

# ==================== 4. 商品维度 ====================
sku_names = [
    '农夫山泉550ml*12瓶', '怡宝纯净水350ml*24瓶', '可口可乐330ml*24罐', '百事可乐330ml*24罐',
    '蒙牛纯牛奶250ml*24盒', '伊利安慕希205g*12盒', '康师傅红烧牛肉面袋装*24', '统一老坛酸菜面*24',
    '奥利奥饼干116g*24盒', '乐事薯片70g*24袋', '海飞丝去屑洗发水750ml', '潘婷洗发水750ml',
    '舒肤佳香皂115g*72块', '蓝月亮洗衣液3kg*4桶', '立白洗洁精1.5kg*6瓶', '红牛维生素饮料250ml*24罐',
    '脉动维生素饮料600ml*15瓶', '元气森林气泡水480ml*15瓶', '三只松鼠坚果750g*10袋', '良品铺子猪肉脯200g*20袋',
    '金龙鱼食用油5L*4桶', '福临门大米5kg*10袋', '心相印抽纸3层*24包', '维达卷纸4层*27卷',
    '雕牌洗洁精1.5kg*6瓶', '汰渍洗衣粉2kg*6袋', '奥妙洗衣液2kg*6瓶', '舒客牙膏120g*12支',
    '云南白药牙膏180g*6支', '百雀羚面霜50g*12瓶', '达利园软面包360g*12袋', '好吃点饼干108g*24盒',
    '旺旺雪饼520g*12袋', '盼盼小面包400g*10袋', '清风抽纸3层*24包', '洁柔卷纸4层*27卷',
    '威猛先生洁厕液500ml*12瓶', '绿伞洗衣液2kg*6瓶', '白猫洗洁精1.5kg*6瓶', '六神花露水195ml*12瓶',
    '大宝SOD蜜100ml*12瓶', '郁美净儿童霜25g*12盒', '海天酱油500ml*12瓶', '李锦记蚝油700g*12瓶',
    '老干妈辣椒酱210g*24瓶', '太太乐鸡精200g*20袋', '恒顺香醋500ml*12瓶', '厨邦酱油410ml*12瓶',
    '鲁花花生油5L*4桶', '西王玉米油5L*4桶',
]
sku_names = sku_names[:NUM_SKU]

sku_list = []
for i in range(1, NUM_SKU + 1):
    sku_id = 1000 + i
    name = sku_names[i-1]
    brand_id = random.randint(1, 5)
    cat_l1 = random.choice([1, 2, 3])
    if cat_l1 == 1:
        cat_l2 = random.choice([10101, 10102])
        cat_l3 = random.choice([1010101, 1010102, 1010201])
    elif cat_l1 == 2:
        cat_l2 = random.choice([10201, 10202])
        cat_l3 = random.choice([1020101, 1020201])
    else:
        cat_l2 = random.choice([10301, 10302])
        cat_l3 = random.choice([1030101, 1030201])
    spec = random.choice(['500ml*12瓶', '1L*6瓶', '250ml*24盒', '100g*24包', '2kg*6袋', '5L*4桶'])
    unit = random.choice(['箱', '包', '瓶', '桶'])
    unit_cost = round(random.uniform(5, 200), 2)
    supplier_id = random.randint(1, 5)
    shelf_life = random.randint(30, 720)
    is_perishable = 1 if shelf_life < 180 else 0
    sku_status = random.choices([1, 2, 0], weights=[0.8, 0.15, 0.05])[0]
    created_time = '2026-01-01 00:00:00'
    updated_time = '2026-09-04 00:00:00'
    etl_time = dt2s(datetime.datetime.now())
    dt = d2s(SNAPSHOT_DATE)
    add_sql(f"INSERT INTO dim_sku (sku_id, sku_name, sku_code, brand_id, category_l1_id, category_l2_id, category_l3_id, spec, unit, unit_cost, supplier_id, shelf_life_days, is_perishable, sku_status, created_time, updated_time, etl_time, dt) VALUES ({sku_id}, '{name}', {sku_id}, {brand_id}, {cat_l1}, {cat_l2}, {cat_l3}, '{spec}', '{unit}', {unit_cost}, {supplier_id}, {shelf_life}, {is_perishable}, {sku_status}, '{created_time}', '{updated_time}', '{etl_time}', '{dt}')")
    sku_list.append(sku_id)

# ==================== 5. 补货规则 ====================
rule_dict = {}

for sku_id in sku_list:
    for wh_id in range(1, NUM_WAREHOUSE + 1):
        base_adu = random.randint(10, 200)
        lead_time = random.randint(3, 15)
        review_cycle = random.choice([7, 14, 30])
        safety_stock = base_adu * random.randint(3, 7)
        target_dos = random.randint(30, 60)
        min_order = random.randint(20, 100)
        max_order = random.randint(500, 2000)
        max_stock = int(base_adu * target_dos * random.uniform(1.2, 1.5))
        rounding = 12
        target_stock = int(base_adu * target_dos)
        buffer = random.randint(1, 5)
        service_level = round(random.uniform(0.9, 0.99), 4)
        demand_days = 30
        avg_daily_sales = base_adu
        model = random.choice(['MIN-MAX', 'ROP', 'PERIODIC'])
        reorder_point = lead_time * base_adu + safety_stock
        forecast_ver = 'v1'
        health_low = 7
        health_high = target_dos
        is_active = 1
        start = '2026-01-01'
        end = '9999-12-31'
        etl_time = dt2s(datetime.datetime.now())

        rule_dict[(sku_id, wh_id)] = {
            'base_adu': base_adu,
            'lead_time': lead_time,
            'safety_stock': safety_stock,
            'target_stock': target_stock,
            'max_stock': max_stock,
            'reorder_point': reorder_point,
            'target_dos': target_dos,
            'review_cycle': review_cycle,
        }

        add_sql(f"INSERT INTO dim_sku_replenishment_rule (sku_id, warehouse_id, lead_time_days, review_cycle_days, safety_stock_qty, target_days_of_supply, min_order_qty, max_order_qty, max_stock_qty, rounding_qty, target_stock_qty, lead_time_buffer_days, service_level, demand_days, avg_daily_sales_qty, replenishment_model, reorder_point_qty, forecast_version, health_days_low, health_days_high, is_active, effective_start_date, effective_end_date, etl_time) VALUES ({sku_id}, {wh_id}, {lead_time}, {review_cycle}, {safety_stock}, {target_dos}, {min_order}, {max_order}, {max_stock}, {rounding}, {target_stock}, {buffer}, {service_level}, {demand_days}, {avg_daily_sales}, '{model}', {reorder_point}, '{forecast_ver}', {health_low}, {health_high}, {is_active}, '{start}', '{end}', '{etl_time}')")

# ==================== 6. 当前库存快照 + 健康状态分配（日期为 SNAPSHOT_DATE） ====================
statuses = ['urgent', 'rop', 'watch', 'normal', 'overstock'] * (NUM_SKU * NUM_WAREHOUSE // 5)
random.shuffle(statuses)

snapshot_id = 1
current_available_dict = {}
state_dict = {}

idx = 0
for sku_id in sku_list:
    for wh_id in range(1, NUM_WAREHOUSE + 1):
        state = statuses[idx]
        idx += 1
        r = rule_dict[(sku_id, wh_id)]
        base_adu = r['base_adu']
        safety_stock = r['safety_stock']
        rop = r['reorder_point']
        target_stock = r['target_stock']
        max_stock = r['max_stock']

        if state == 'urgent':
            available = int(safety_stock * random.uniform(0.3, 0.8))
        elif state == 'rop':
            available = int(rop * random.uniform(0.9, 1.1))
        elif state == 'watch':
            watch_level = rop + 3 * base_adu
            available = int(watch_level * random.uniform(0.9, 1.1))
        elif state == 'normal':
            available = int(target_stock * random.uniform(0.8, 1.2))
        else:  # overstock
            available = int(max_stock * random.uniform(1.1, 1.5))

        available = max(0, available)
        locked = int(available * random.uniform(0.05, 0.15))
        on_hand = available + locked
        in_transit = random.randint(0, 200)
        reserved = random.randint(0, 20)
        damaged = random.randint(0, 5)
        frozen = random.randint(0, 10)
        data_source = random.choice(['oms', 'wms', 'manual'])
        etl_time = dt2s(datetime.datetime.now())
        dt = d2s(SNAPSHOT_DATE)
        snapshot_time = dt2s(datetime.datetime.combine(SNAPSHOT_DATE, datetime.time(8, 0, 0)))

        add_sql(f"INSERT INTO dwd_inventory_snapshot (snapshot_id, snapshot_time, sku_id, warehouse_id, on_hand_qty, locked_qty, available_qty, in_transit_qty, reserved_qty, damaged_qty, frozen_qty, data_source, etl_time, dt) VALUES ({snapshot_id}, '{snapshot_time}', {sku_id}, {wh_id}, {on_hand}, {locked}, {available}, {in_transit}, {reserved}, {damaged}, {frozen}, '{data_source}', '{etl_time}', '{dt}')")

        current_available_dict[(sku_id, wh_id)] = available
        state_dict[(sku_id, wh_id)] = state
        snapshot_id += 1

# ==================== 7. 历史每日销售数据（日期范围 START_DATE ~ END_DATE_HIST） ====================
for day in range(NUM_DAYS):
    date = START_DATE + datetime.timedelta(days=day)
    dt_str = d2s(date)
    weekday = date.weekday()
    for (sku_id, wh_id), r in rule_dict.items():
        base_adu = r['base_adu']
        adu_factor = 1.2 if weekday >= 5 else 1.0
        is_promo = 1 if random.random() < 0.1 else 0
        promo_factor = 2.0 if is_promo else 1.0
        expected_sales = base_adu * adu_factor * promo_factor
        sales_qty = max(0, int(random.gauss(expected_sales, expected_sales ** 0.5)))
        return_qty = int(sales_qty * random.uniform(0, 0.05))
        net_qty = max(0, sales_qty - return_qty)
        price = round(random.uniform(5, 200), 2)
        sales_amt = round(net_qty * price, 2)
        return_amt = round(return_qty * price, 2)
        order_count = max(1, net_qty // 2)
        shipment_qty = net_qty
        avg_price = price
        promo_out_qty = sales_qty if is_promo else 0
        exclude_flag = 0
        etl_time = dt2s(datetime.datetime.now())
        add_sql(f"INSERT INTO dws_act_sales_daily (dt, sku_id, channel_id, warehouse_id, sales_qty, sales_amt, return_qty, return_amt, net_sales_qty, net_sales_amt, order_count, shipment_qty, avg_price, is_promotion, promo_out_qty, exclude_cal_flag, etl_time) VALUES ('{dt_str}', {sku_id}, 1, {wh_id}, {sales_qty}, {sales_amt}, {return_qty}, {return_amt}, {net_qty}, {sales_amt}, {order_count}, {shipment_qty}, {avg_price}, {is_promo}, {promo_out_qty}, {exclude_flag}, '{etl_time}')")

# ==================== 8. 历史每日库存数据（日期范围 START_DATE ~ END_DATE_HIST） ====================
for (sku_id, wh_id), current_avail in current_available_dict.items():
    state = state_dict[(sku_id, wh_id)]
    r = rule_dict[(sku_id, wh_id)]
    base_adu = r['base_adu']
    safety_stock = r['safety_stock']
    rop = r['reorder_point']
    target_stock = r['target_stock']
    max_stock = r['max_stock']

    if state == 'urgent':
        start_avail = current_avail + int(base_adu * random.randint(20, 40))
        trend = 'down'
    elif state == 'rop':
        start_avail = rop + int(base_adu * random.randint(10, 20))
        trend = 'down'
    elif state == 'watch':
        watch_level = rop + 3 * base_adu
        start_avail = watch_level + int(base_adu * random.randint(5, 10))
        trend = 'stable'
    elif state == 'normal':
        start_avail = target_stock * random.uniform(0.9, 1.1)
        trend = 'stable'
    else:
        start_avail = current_avail * random.uniform(0.9, 1.1)
        trend = 'stable_high'

    daily_avail = []
    for day in range(NUM_DAYS):
        if trend == 'down':
            factor = (NUM_DAYS - 1 - day) / (NUM_DAYS - 1) if NUM_DAYS > 1 else 1
            avail = int(start_avail * factor + current_avail * (1 - factor))
        elif trend == 'stable':
            noise = random.gauss(0, base_adu * 2)
            avail = int(target_stock + noise)
            if day == NUM_DAYS - 1:
                avail = current_avail
        elif trend == 'stable_high':
            noise = random.gauss(0, base_adu * 2)
            avail = int(current_avail + noise)
            if day == NUM_DAYS - 1:
                avail = current_avail
        else:
            avail = current_avail
        avail = max(0, avail)
        daily_avail.append(avail)

    # 强制最后一天等于当前快照库存
    daily_avail[-1] = current_avail

    for day in range(NUM_DAYS):
        date = START_DATE + datetime.timedelta(days=day)
        dt_str = d2s(date)
        avail = daily_avail[day]
        locked = int(avail * random.uniform(0.05, 0.15))
        on_hand = avail + locked
        begin_on_hand = on_hand
        begin_locked = locked
        begin_avail = avail
        end_on_hand = on_hand
        end_locked = locked
        end_avail = avail
        defect = random.randint(0, 5)
        in_transit = random.randint(0, 50)
        max_onhand = int(on_hand * random.uniform(1.0, 1.1))
        min_onhand = int(on_hand * random.uniform(0.9, 1.0))
        avg_on_hand = (begin_on_hand + end_on_hand) // 2
        avg_avail = (begin_avail + end_avail) // 2
        unit_cost = random.uniform(5, 200)
        inventory_cost_amt = round(end_avail * unit_cost, 2)
        turnover_days = int(avg_avail / max(1, base_adu)) if base_adu > 0 else 0
        stock_age = random.randint(1, 180)
        etl_time = dt2s(datetime.datetime.now())
        add_sql(f"INSERT INTO dws_fact_inventory (dt, sku_id, warehouse_id, begin_on_hand_qty, begin_locked_qty, begin_available_qty, end_on_hand_qty, end_locked_qty, end_available_qty, defect_qty, in_transit_qty, max_onhand_qty, min_onhand_qty, avg_on_hand_qty, avg_available_qty, inventory_cost_amt, turnover_days, stock_age_days, etl_time) VALUES ('{dt_str}', {sku_id}, {wh_id}, {begin_on_hand}, {begin_locked}, {begin_avail}, {end_on_hand}, {end_locked}, {end_avail}, {defect}, {in_transit}, {max_onhand}, {min_onhand}, {avg_on_hand}, {avg_avail}, {inventory_cost_amt}, {turnover_days}, {stock_age}, '{etl_time}')")

# ==================== 9. 采购在途事实表 ====================
po_id = 9000
po_line_id = 1
low_stock_combos = [(s, w) for (s, w), st in state_dict.items() if st in ['urgent', 'rop', 'watch']]
normal_combos = [(s, w) for (s, w), st in state_dict.items() if st in ['normal', 'overstock']]

for _ in range(40):
    if random.random() < 0.7 and low_stock_combos:
        sku_id, wh_id = random.choice(low_stock_combos)
    else:
        sku_id, wh_id = random.choice(normal_combos if normal_combos else low_stock_combos)

    supplier_id = random.randint(1, 5)
    po_date = (SNAPSHOT_DATE - datetime.timedelta(days=random.randint(0, 15))).strftime('%Y-%m-%d')
    po_qty = random.randint(100, 1000)
    received = random.randint(0, po_qty // 2)
    onway = po_qty - received
    expect_date = (SNAPSHOT_DATE + datetime.timedelta(days=random.randint(5, 25))).strftime('%Y-%m-%d')
    expected_7d = onway if random.random() < 0.3 else 0
    expected_30d = onway - expected_7d
    actual_arrival = 'NULL'
    lead_time = random.randint(3, 15)
    unit_price = round(random.uniform(5, 200), 2)
    po_amt = round(po_qty * unit_price, 2)
    po_status = random.choice([1, 2, 2, 2])
    is_valid_onway = 1 if po_status != 4 else 0
    etl_time = dt2s(datetime.datetime.now())
    dt = d2s(SNAPSHOT_DATE)
    create_dt = po_date

    add_sql(f"INSERT INTO dws_fact_po_onway (dt, po_id, po_line_id, sku_id, warehouse_id, supplier_id, po_date, po_qty, received_qty, onway_qty, expect_arrive_date, expected_arrive_qty_7d, expected_arrive_qty_30d, earliest_expected_date, latest_expected_date, actual_arrival_date, lead_time_days, create_dt, unit_price, po_amt, po_status, is_valid_onway, etl_time) VALUES ('{dt}', {po_id}, {po_line_id}, {sku_id}, {wh_id}, {supplier_id}, '{po_date}', {po_qty}, {received}, {onway}, '{expect_date}', {expected_7d}, {expected_30d}, '{expect_date}', '{expect_date}', {actual_arrival}, {lead_time}, '{create_dt}', {unit_price}, {po_amt}, {po_status}, {is_valid_onway}, '{etl_time}')")
    po_id += 1
    po_line_id += 1

# ==================== 写出文件 ====================
with open('mock_data.sql', 'w', encoding='utf-8') as f:
    f.write('USE Supply_Chain;\n')
    f.write('SET FOREIGN_KEY_CHECKS=0;\n')
    f.write('\n'.join(sql_lines))
    f.write('\nSET FOREIGN_KEY_CHECKS=1;\n')

print(f"模拟数据生成完成，共 {len(sql_lines)} 条 INSERT 语句。")
print("文件保存为 mock_data.sql")

模拟数据生成完成，共 20214 条 INSERT 语句。
文件保存为 mock_data.sql
